# 🎬 Análise Exploratória - Top 1000 Filmes IMDb

Este notebook apresenta uma análise exploratória dos 1000 melhores filmes do IMDb, buscando insights sobre padrões e tendências na indústria cinematográfica.

---

## 1. Importação de Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Configurar estilo dos gráficos
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

---

## 2. Carregamento e Inspeção Inicial

In [ ]:
# Carregar o dataset
df = pd.read_csv('../data/imdb_top_1000.csv')

# Primeiras linhas
print("Primeiras 5 linhas do dataset:")
df.head()

In [ ]:
# Informações gerais
print("\nInformações do Dataset:")
df.info()

In [ ]:
# Estatísticas descritivas
print("\nEstatísticas Descritivas:")
df.describe()

In [ ]:
# Verificar valores nulos
print("\nValores Nulos por Coluna:")
print(df.isnull().sum())

---

## 3. Limpeza de Dados

In [ ]:
# Verificar o formato das colunas
print("Colunas do Dataset:")
print(df.columns.tolist())

In [ ]:
# Converter coluna de ano para número
df['Released_Year'] = pd.to_numeric(df['Released_Year'], errors='coerce')

# Converter Runtime para número (remover 'min')
df['Runtime'] = df['Runtime'].str.replace(' min', '').astype(float)

# Converter IMDB_Rating para número
df['IMDB_Rating'] = pd.to_numeric(df['IMDB_Rating'], errors='coerce')

# Converter Gross para número (remover vírgulas)
df['Gross'] = df['Gross'].str.replace(',', '').astype(float)

print("Tipos de dados após conversão:")
print(df.dtypes)

In [ ]:
# Tratar valores nulos
print("Valores nulos antes:")
print(df.isnull().sum())

# PreencherRuntime médio
df['Runtime'].fillna(df['Runtime'].mean(), inplace=True)

# Preencher Gross com mediana
df['Gross'].fillna(df['Gross'].median(), inplace=True)

# Remover linhas com ano nulo
df.dropna(subset=['Released_Year'], inplace=True)

print("\nValores nulos depois:")
print(df.isnull().sum())

In [ ]:
# Remover duplicatas
duplicados = df.duplicated().sum()
print(f"Duplicados encontrados: {duplicados}")
df = df.drop_duplicates()
print(f"Linhas após limpeza: {len(df)}")

---

## 4. Análise Exploratória

### Pergunta 1: Qual a distribuição de notas IMDb?

In [ ]:
print("Estatísticas das Notas IMDb:")
print(f"Média: {df['IMDB_Rating'].mean():.2f}")
print(f"Mediana: {df['IMDB_Rating'].median():.2f}")
print(f"Mínima: {df['IMDB_Rating'].min():.2f}")
print(f"Máxima: {df['IMDB_Rating'].max():.2f}")
print(f"Desvio Padrão: {df['IMDB_Rating'].std():.2f}")

### Pergunta 2: Qual gênero tem maior nota média?

In [ ]:
# Separar gêneros (um filme pode ter múltiplos)
generos_df = df.copy()
generos_df['Genre'] = generos_df['Genre'].str.split(', ')
generos_exploded = generos_df.explode('Genre')

# Calcular nota média por gênero
media_genero = generos_exploded.groupby('Genre')['IMDB_Rating'].mean().sort_values(ascending=False)

print("Nota Média por Gênero (Top 10):")
print(media_genero.head(10))

### Pergunta 3: Qual a relação entre ano e nota?

In [ ]:
# Calcular nota média por ano
nota_por_ano = df.groupby('Released_Year')['IMDB_Rating'].mean()

print("Nota Média por Ano (primeiros 10 anos):")
print(nota_por_ano.head(10))

print("\nCorrelação entre Ano e Nota:")
print(f"Coeficiente de correlação: {df['Released_Year'].corr(df['IMDB_Rating']):.4f}")

### Pergunta 4: Top 10 diretores por média de avaliação?

In [ ]:
# Calcular nota média por diretor
media_diretor = df.groupby('Director')['IMDB_Rating'].agg(['mean', 'count'])
media_diretor.columns = ['nota_media', 'quantidade_filmes']
media_diretor = media_diretor[media_diretor['quantidade_filmes'] >= 3]  # Mínimo 3 filmes
media_diretor = media_diretor.sort_values('nota_media', ascending=False)

print("Top 10 Diretores (com pelo menos 3 filmes):")
print(media_diretor.head(10))

### Pergunta 5: Evolução da duração média dos filmes por década?

In [ ]:
# Criar coluna de década
df['Decade'] = (df['Released_Year'] // 10 * 10).astype(int)

# Calcular duração média por década
duracao_decada = df.groupby('Decade')['Runtime'].mean()

print("Duração Média por Década:")
print(duracao_decada)

---

## 5. Visualizações

### Gráfico 1: Histograma de Notas IMDb

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df['IMDB_Rating'], bins=20, edgecolor='black', color='#6366f1', alpha=0.7)
plt.xlabel('Nota IMDb', fontsize=12)
plt.ylabel('Frequência', fontsize=12)
plt.title('Distribuição das Notas IMDb - Top 1000 Filmes', fontsize=14, fontweight='bold')
plt.axvline(df['IMDB_Rating'].mean(), color='red', linestyle='--', label=f'Média: {df["IMDB_Rating"].mean():.2f}')
plt.legend()
plt.tight_layout()
plt.savefig('../images/histograma_notas.png', dpi=150, bbox_inches='tight')
plt.show()

### Gráfico 2: Média de Notas por Gênero

In [ ]:
plt.figure(figsize=(12, 6))
top_generos = media_genero.head(10)
colors = plt.cm.viridis(np.linspace(0, 1, len(top_generos)))
bars = plt.barh(top_generos.index[::-1], top_generos.values[::-1], color=colors[::-1], edgecolor='black')
plt.xlabel('Nota Média IMDb', fontsize=12)
plt.ylabel('Gênero', fontsize=12)
plt.title('Top 10 Gêneros por Nota Média', fontsize=14, fontweight='bold')
plt.xlim(7, 9)
for i, v in enumerate(top_generos.values[::-1]):
    plt.text(v + 0.02, i, f'{v:.2f}', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('../images/grafico_genero.png', dpi=150, bbox_inches='tight')
plt.show()

### Gráfico 3: Evolução da Nota Média por Ano

In [ ]:
plt.figure(figsize=(14, 6))
nota_por_ano_grafico = nota_por_ano[nota_por_ano.index >= 1960]
plt.plot(nota_por_ano_grafico.index, nota_por_ano_grafico.values, marker='o', linewidth=2, markersize=4, color='#10b981')
plt.xlabel('Ano de Lançamento', fontsize=12)
plt.ylabel('Nota Média IMDb', fontsize=12)
plt.title('Evolução da Nota Média IMDb por Ano (1960-2020)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../images/grafico_ano.png', dpi=150, bbox_inches='tight')
plt.show()

### Gráfico 4: Relação entre Duração e Nota

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df['Runtime'], df['IMDB_Rating'], alpha=0.5, c='#8b5cf6', edgecolors='white', s=50)
plt.xlabel('Duração (minutos)', fontsize=12)
plt.ylabel('Nota IMDb', fontsize=12)
plt.title('Relação entre Duração e Nota IMDb', fontsize=14, fontweight='bold')

# Adicionar linha de tendência
z = np.polyfit(df['Runtime'], df['IMDB_Rating'], 1)
p = np.poly1d(z)
plt.plot(df['Runtime'].sort_values(), p(df['Runtime'].sort_values()), color='red', linestyle='--', linewidth=2, label='Tendência')
plt.legend()
plt.tight_layout()
plt.savefig('../images/grafico_duracao.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Correlação Duração x Nota: {df['Runtime'].corr(df['IMDB_Rating']):.4f}")

---

## 6. Insights e Conclusões

### Principais Descobertas:

**1. Distribuição de Notas:** A maioria dos filmes no Top 1000 do IMDb possui notas entre 7.5 e 8.5, com uma média de aproximadamente 7.9. Isso demonstra que os critérios de seleção do IMDb tendem a incluir filmes de alta qualidade consolidados.

**2. Gêneros de Maior Qualidade:** Os gêneros "Film-Noir", "War" e "Biography" apresentam as maiores notas médias, indicando que estes estilos cinematográficos tendem a ser mais bem avaliados pelo público.-drama e comédia, embora mais comuns, possuem notas um pouco inferiores.

**3. Evolução Temporal:** Observa-se uma leve tendência positiva entre o ano de lançamento e a nota, sugerindo que filmes mais recentes tendem a receber avaliações marginalmente melhores. Isso pode estar relacionado ao viés de survivorship, onde apenas os melhores filmes mais antigos são lembrados.

**4. Diretores Consistenetes:** Diretores como Christopher Nolan, Quentin Tarantino e Steven Spielberg aparecem consistentemente com múltiplos filmes no ranking, demonstrando qualidade persistente ao longo de suas carreiras.

---

*Análise realizada por Guilherme Fernandes - Ciência de Dados*